# 08 NLP Evaluation Metrics: Classification, BLEU, ROUGE & Perplexity

**Real-World Scenario**: Enterprise SLA Documentation Automatic Summarization Evaluation System.

This notebook demonstrates computing BLEU-1 through BLEU-4, ROUGE-1/2/L, Perplexity from Cross-Entropy loss, saving metric evaluation logs into a hidden `.model_cache/` directory, and executing a **Complete GenAI Metric Audit LLM Call**.

## Step 1: Environment Setup & Local Cache Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from root .env file
load_dotenv()

# Create hidden model cache directory for local pipeline artifact persistence
os.makedirs(".model_cache", exist_ok=True)

print("=== Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden Cache Directory Path:", os.path.abspath(".model_cache"))

## Step 2: Realistic Candidate vs Reference Enterprise Summary Dataset Setup

In [ ]:
reference_summary = "Cloud infrastructure downtime exceeding 0.05% SLA triggers automatic financial penalty refunds within 30 business days."
candidate_summary = "Downtime exceeding 0.05% SLA triggers automatic financial penalty refunds in 30 days."

print("Reference Summary: ", reference_summary)
print("Candidate Summary: ", candidate_summary)

## Step 3: Computing BLEU & ROUGE Evaluation Metrics

In [ ]:
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

ref_tokens = reference_summary.split()
cand_tokens = candidate_summary.split()

smooth_fn = SmoothingFunction().method1
bleu_1 = sentence_bleu([ref_tokens], cand_tokens, weights=(1, 0, 0, 0), smoothing_function=smooth_fn)
bleu_2 = sentence_bleu([ref_tokens], cand_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth_fn)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_scores = scorer.score(reference_summary, candidate_summary)

print("=== Computed Automated NLP Metrics ===")
print(f"BLEU-1 Precision:  {bleu_1:.4f}")
print(f"BLEU-2 Precision:  {bleu_2:.4f}")
print(f"ROUGE-1 Recall:    {rouge_scores['rouge1'].recall:.4f}")
print(f"ROUGE-2 Recall:    {rouge_scores['rouge2'].recall:.4f}")
print(f"ROUGE-L F1-Score:  {rouge_scores['rougeL'].fmeasure:.4f}")

## Step 4: Computing Language Model Perplexity & Metrics Cache Persistence

In [ ]:
eval_cross_entropy_loss = 1.4500
perplexity = np.exp(eval_cross_entropy_loss)

print(f"Cross-Entropy Loss: {eval_cross_entropy_loss:.4f} -> Perplexity PPL: {perplexity:.4f}")

# Save evaluation log artifact into hidden cache folder
log_path = os.path.join(".model_cache", "eval_metrics_summary.txt")
with open(log_path, "w") as f:
    f.write(f"BLEU-1: {bleu_1:.4f}\nROUGE-1: {rouge_scores['rouge1'].recall:.4f}\nPPL: {perplexity:.4f}\n")

print(f"=== Evaluation Metrics Log Persistence Saved to Hidden Folder: {log_path} ===")

## Step 5: Executing Complete GenAI Metric Audit LLM Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

if os.getenv("OPENAI_API_KEY"):
    prompt = ChatPromptTemplate.from_template('''You are an Enterprise AI Model Quality Assurance Auditor.
Audit the candidate generated text quality given the automated metrics below.

Metrics:
- BLEU-1: {bleu1:.4f}
- ROUGE-1 Recall: {rouge1:.4f}
- Language Model Perplexity: {ppl:.4f}

Candidate Summary: '{candidate}'

Quality Audit Assessment:''')
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    
    response = llm.invoke(prompt.format(
        bleu1=bleu_1, 
        rouge1=rouge_scores['rouge1'].recall, 
        ppl=perplexity, 
        candidate=candidate_summary
    ))
    print("=== Complete GenAI Quality Audit Response ===")
    print(response.content)
else:
    print("[NOTICE] OPENAI_API_KEY not found in .env. Skipping live LLM call.")